In [ ]:
# nanoGPT from scratch on Kaggle
# 第 1 个 cell：环境检查、目标说明、导入依赖、固定随机种子
#
# 这份 notebook 的目标不是直接调用官方 nanoGPT 的 train.py，
# 而是在 Kaggle 里一步一步“自己写出一个 nanoGPT”。
# 官方源码已经放在仓库的 nanoGPT-reference/ 目录中，后续每一步都会对照它来实现。
#
# 当前第 1 格只做准备工作：
# 1. 确认 PyTorch 和 GPU 状态
# 2. 导入后续会用到的基础库
# 3. 固定随机种子，方便复现实验
# 4. 设置一个 device 变量，后续张量和模型都放到这个设备上

import math
import os
import random
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F


print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = "cuda"
else:
    print("GPU not found, using CPU. 也能跑小实验，但会慢一些。")
    device = "cpu"


# 固定随机种子。
# 训练神经网络时，初始化权重、打乱数据、采样 token 都会用到随机数。
# 固定 seed 后，每次从头运行 notebook，结果更容易复现和对比。
seed = 1337
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)


# 允许 TensorFloat-32。它是 NVIDIA GPU 上的一种加速格式。
# 对我们这种学习实验来说，可以让矩阵乘法更快。
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


print("Using device:", device)
print("Seed:", seed)


# 后续我们会按这个顺序继续写：
# Cell 2：准备一个极小文本数据集，并把字符变成整数 token
# Cell 3：写 get_batch，构造 x/y 训练样本
# Cell 4：先实现 BigramLanguageModel，理解最小语言模型
# Cell 5：实现单头 causal self-attention
# Cell 6：实现多头注意力
# Cell 7：实现 MLP 和 Block
# Cell 8：组装 GPTConfig 和 GPT
# Cell 9：写训练循环
# Cell 10：写 generate 生成文本
